# Chambolle-Pock for Salt-and-Pepper Denoising with TV-L1

In this notebook, we demonstrate the Chambolle-Pock method on a second kind of noise: salt and pepper, where a fraction of the pixels of a grayscale image is replaced by pure black or pure white.

The color notebook solves the Rudin-Osher-Fatemi (ROF) model, whose quadratic data term suits Gaussian noise. Salt-and-pepper noise is different: most pixels are exact, and the rest are as wrong as they can be. Chambolle and Pock [1] treat this case with the TV-L1 model, which keeps the total variation of ROF and replaces its quadratic data term with an $\ell_1$ norm.

We do not write a new solver. A pixel grid is a graph, so the same `chambolle_pock` that denoises point clouds runs on images with a different set of edges and a different data term.

Particularly, we consider the following problem:
- Start from the ROF model and replace its data term with the $\ell_1$ norm.
- Write a grayscale image as a graph so that the point cloud solver applies unchanged.
- Change the primal step of the Chambolle-Pock method to the proximal map of the $\ell_1$ term.
- Compare TV-L1 with ROF on salt-and-pepper noise, and analyze the effect of the parameters.

## Problem setup

We use three clean grayscale images in $[0, 1]$:
- A synthetic piecewise constant image: a square, a disk, and a thin bar on a flat background (`synthetic_image`).
- The Louvre pyramid, the test image of Figure 4(a) in [1] (500 × 375).
- The public-domain NASA portrait of the astronaut Eileen Collins that ships with scikit-image, read as grayscale.

Each image is shrunk so that its longer side is at most a given number of pixels. A fraction $p = 0.25$ of the pixels is then picked at random, and each picked pixel is set to 0 or 1 with equal chance:

$$
f_i =
\begin{cases}
0 & \text{with probability } p/2, \\
1 & \text{with probability } p/2, \\
u^{\mathrm{clean}}_i & \text{with probability } 1 - p.
\end{cases}
$$

The images are in `models/experiments/exp_tvl1/`, and the noise is `add_salt_and_pepper` in `experiments/exp_tvl1/image.py`.

In [ ]:
# The test images are located in `models/experiments/exp_tvl1/`: `louvre.png` from Figure 4 of [1] and `astronaut.png` from scikit-image.

%matplotlib widget
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

ROOT = Path.cwd()
if not (ROOT / "experiments" / "exp_tvl1").exists():
    ROOT = ROOT.parent
sys.path[:0] = [str(ROOT / "src"), str(ROOT)]

from cp_pointcloud import denoise_image, tvl1_energy
from experiments.exp_tvl1.figure import draw_gray_image
from experiments.exp_tvl1.image import add_salt_and_pepper, image_mse, read_grayscale, synthetic_image

IMAGE_DIR = ROOT / "models" / "experiments" / "exp_tvl1"
IMAGE_NAMES = ["synthetic", "louvre", "astronaut"]
SIZE = 128
FRACTION = 0.25
SEED = 0


def load_image(name, size):
    if name == "synthetic":
        return synthetic_image(size)
    return read_grayscale(IMAGE_DIR / f"{name}.png", size)


def make_noisy(clean, fraction):
    return add_salt_and_pepper(clean, fraction, np.random.default_rng(SEED))


clean = load_image("synthetic", SIZE)
noisy = make_noisy(clean, FRACTION)

# Plot the clean and noisy versions of every test image
figure, axes = plt.subplots(2, 3, figsize=(11, 7))
for column, name in enumerate(IMAGE_NAMES):
    image = load_image(name, 256)
    draw_gray_image(axes[0, column], image, f"{name}, clean")
    draw_gray_image(axes[1, column], make_noisy(image, FRACTION), f"{name}, noisy")
figure.tight_layout()
plt.show()

## From ROF to TV-L1

The color notebook solves the ROF problem

$$
\min_u \frac{\lambda}{2}\|u - f\|_2^2 + \|\nabla u\|_1 ,
$$

where $f$ is the noisy image. The TV-L1 model of [1] keeps the total variation and changes the data term:

$$
\min_u \|\nabla u\|_1 + \lambda \|u - f\|_1 .
$$

1. Why an $\ell_1$ data term:
The quadratic term charges a residual $r = u_i - f_i$ the amount $r^2$, so its pull on $u_i$ grows with the size of the residual. An impulse that took a pixel from 0.2 to 1 pulls hard, and ROF moves the restored value part of the way toward it. The impulse is not removed but spread over its neighborhood. The $\ell_1$ term charges $|r|$. Its pull is $\pm\lambda$, whatever the size of the residual, so an impulse pulls no harder than a small error, and it is cheaper to ignore the impulse entirely. This is the reason the median is robust to outliers while the mean is not: $\sum_i (u - f_i)^2$ is minimized by the mean, and $\sum_i |u - f_i|$ by the median.

2. Contrast does not decide what is removed:
Consider a flat feature of height $h$ above its background, with area $A$ and perimeter $P$. Keeping it costs about $hP$ in total variation. Removing it costs $\lambda h A$ in the data term. Both are proportional to $h$, so TV-L1 removes the feature when $P > \lambda A$, whatever its contrast [3]. Single-pixel impulses have a large perimeter for their area, so they are removed. Large shapes are kept with their exact gray value. ROF has no such threshold, since its data cost $\frac{\lambda}{2} h^2 A$ grows faster than the total variation. It lowers the contrast of every feature a little instead.

3. The parameter $\lambda$:
A larger $\lambda$ keeps the restored image closer to the noisy one. A smaller $\lambda$ removes larger structures. The $\lambda$ of TV-L1 and the $\lambda$ of ROF have different units, so their values cannot be compared directly.

The TV-L1 problem is convex but not strictly convex. At the threshold above, keeping and removing a feature cost the same, and both are solutions.

## The pixel grid as a graph

In the color notebook, the point cloud is a $k$-nearest-neighbor graph, and the gradient on an edge is the weighted difference $(\nabla u)_{ij} = \sqrt{w_{ij}}\,(u_j - u_i)$. A pixel grid is a special case of such a graph. Each pixel $(r, c)$ has one edge to its right neighbor $(r, c+1)$ and one to the pixel below $(r+1, c)$, each of weight $w = 1$. Unlike the $k$-nearest-neighbor graph, the edges are not symmetrized, so they give forward differences.

The norm at a vertex stacks the differences along its outgoing edges:

$$
|(\nabla u)_{r,c}| = \sqrt{(u_{r,c+1} - u_{r,c})^2 + (u_{r+1,c} - u_{r,c})^2},
\qquad
\mathrm{TV}(u) = \sum_{r,c} |(\nabla u)_{r,c}|.
$$

This is the discrete isotropic total variation of [1]. A pixel on the last row or column has no edge in that direction, so that difference is 0, which is the Neumann boundary condition of [1].

Every pixel has at most two outgoing and two incoming edges of weight 1. The bound `operator_norm_bound` uses on the graph, $\|K\|^2 \le 4 \max_i \sum_{j \sim i} w_{ij}$, then gives $\|K\|^2 \le 8$, the same bound as [1].

The edges are built by `grid_graph` in `src/cp_pointcloud/tvl1_denoise.py`. `denoise_image` passes them to the same `chambolle_pock` in `src/cp_pointcloud/denoise.py` that the point clouds use, with `data="l1"`.

## The Chambolle–Pock iteration for TV-L1

As before, we write the problem as $\min_x F(Kx) + G(x)$. $K$ is the gradient on the grid, $F$ is the sum of the vertex norms, and now $G(u) = \lambda\|u - f\|_1$. The iteration is unchanged, with $\theta = 1$:

$$
\begin{aligned}
y^{n+1} &= \operatorname{prox}_{\sigma F^*}\bigl(y^n + \sigma K \bar{x}^n\bigr), \\
x^{n+1} &= \operatorname{prox}_{\tau G}\bigl(x^n - \tau K^* y^{n+1}\bigr), \\
\bar{x}^{n+1} &= 2x^{n+1} - x^n .
\end{aligned}
$$

The dual step does not change. $F^*$ is the indicator of the set where every vertex norm is at most 1, so its proximal map is the projection that divides the dual values on the outgoing edges of vertex $i$ by $\max(1, |y_i|)$.

Only the primal step changes. For ROF the proximal map of the quadratic term is a weighted average,

$$
\operatorname{prox}_{\tau G}(\tilde{x})_i = \frac{\tilde{x}_i + \tau\lambda f_i}{1 + \tau\lambda}.
$$

For TV-L1 it soft-thresholds the residual toward the data,

$$
\operatorname{prox}_{\tau G}(\tilde{x})_i = f_i + \operatorname{sign}(\tilde{x}_i - f_i)\,\max\bigl(|\tilde{x}_i - f_i| - \tau\lambda,\ 0\bigr).
$$

A pixel within $\tau\lambda$ of its noisy value snaps exactly onto it, and a pixel farther away moves by $\tau\lambda$ toward it. This is `_data_prox` in `src/cp_pointcloud/denoise.py`.

The step sizes are $\tau = \sigma = 0.99/\sqrt{8}$, so $\tau\sigma\|K\|^2 < 1$. We stop when the relative change of both $x$ and $y$ falls below $10^{-5}$, or after 5000 iterations. Both are needed: the soft threshold can hold $x$ exactly at $f$ while $y$ is still moving, so a test on $x$ alone would stop too early. $G$ is not uniformly convex, so the accelerated variant of [1] does not apply, and we use the basic iteration (Algorithm 1 of [1]).

## A check: one impulse on a flat image

The threshold of the previous section can be computed exactly for a single pixel. Take a flat image of value 0 with one pixel of height $h$ in its interior, and let $u$ be the restored value of that pixel while the others stay at 0. Three vertex norms see it: its own, with two forward differences, gives $\sqrt{u^2 + u^2} = \sqrt{2}\,|u|$. The pixel to its left and the pixel above each have one forward difference into it, and give $|u|$ each. So

$$
E(u) = (2 + \sqrt{2})\,|u| + \lambda\,|u - h| .
$$

On $[0, h]$ this is linear in $u$, so the minimum is at an end. The impulse is removed ($u = 0$) when $\lambda < 2 + \sqrt{2} \approx 3.41$, and kept whole ($u = h$) when $\lambda > 2 + \sqrt{2}$. Nothing in between is ever optimal, and $h$ does not appear in the threshold. ROF shrinks the impulse gradually instead.

Below we sweep $\lambda$ on a 9 × 9 image with an impulse of height 1 at its center, and plot the restored center value. `tests/test_tvl1_denoise.py` checks both sides of the threshold.

In [ ]:
SPIKE_THRESHOLD = 2.0 + np.sqrt(2.0)
spike = np.zeros((9, 9))
spike[4, 4] = 1.0
spike_lams = np.linspace(1.0, 8.0, 57)
spike_tvl1 = [denoise_image(spike, lam, data="l1")[4, 4] for lam in spike_lams]
spike_rof = [denoise_image(spike, lam, data="l2")[4, 4] for lam in spike_lams]

spike_figure, spike_axis = plt.subplots(figsize=(7, 3.8))
spike_axis.plot(spike_lams, spike_tvl1, marker="o", markersize=3, label="TV-L1")
spike_axis.plot(spike_lams, spike_rof, marker="o", markersize=3, label="ROF")
spike_axis.axvline(SPIKE_THRESHOLD, color="0.5", linestyle="--", linewidth=1)
spike_axis.annotate(
    r"$2 + \sqrt{2}$",
    (SPIKE_THRESHOLD, 0.5),
    xytext=(6, 0),
    textcoords="offset points",
    color="0.35",
)
spike_axis.set_xlabel(r"$\lambda$")
spike_axis.set_ylabel("restored center value")
spike_axis.set_ylim(-0.05, 1.05)
spike_axis.legend(frameon=False, loc="lower right")
spike_axis.spines[["top", "right"]].set_visible(False)
spike_figure.tight_layout()
plt.show()

## TV-L1 denoising of salt and pepper

We demonstrate the method on the synthetic image, 128 × 128 with $p = 0.25$, and measure the mean square error against the clean image:

$$
\mathrm{MSE} = \frac{1}{N}\sum_{i=1}^{N}\bigl(u_i - u^{\mathrm{clean}}_i\bigr)^2 .
$$

We also report the TV-L1 energy $E(u) = \mathrm{TV}(u) + \lambda\|u - f\|_1$ (`tvl1_energy`) of the noisy image and of the restored one. The solver should lower it.

Above $\lambda = 2 + \sqrt{2}$, isolated impulses are kept, and the result stays close to the noisy image. Around $\lambda = 1$, the impulses are gone and every shape keeps its exact gray value. Below about $\lambda = 0.6$, the thin bar has too much perimeter for its area and is removed as well. The square and the disk are removed below about $\lambda = 0.1$, close to the estimate $P/A = 4/51 \approx 0.08$ for the 51-pixel square.

Below we show an interactive plot of the denoising result across values of $\lambda$.

In [ ]:
import ipywidgets as widgets
from IPython.display import display

with plt.ioff():
    comparison = plt.figure(figsize=(11, 3.9))
axis_clean, axis_noisy, axis_restored = comparison.subplots(1, 3)
comparison.subplots_adjust(top=0.86, bottom=0.02, wspace=0.08)
draw_gray_image(axis_clean, clean, "Clean")
draw_gray_image(axis_noisy, noisy, "Noisy")

tvl1_cache = {}
caption = widgets.HTML(layout=widgets.Layout(width="100%"))
lam_slider = widgets.FloatLogSlider(
    value=1.0,
    base=10,
    min=-1.5,
    max=1.0,
    step=0.05,
    description="λ",
    continuous_update=False,
    readout_format=".3g",
    layout=widgets.Layout(width="420px"),
    style={"description_width": "16px"},
)


def caption_html(text):
    return f"<div style='text-align:center; font-size:14px; color:#333; padding-top:4px;'>{text}</div>"


def tvl1_caption(restored, lam):
    return caption_html(
        f"MSE &nbsp; noisy {image_mse(noisy, clean):.3e}"
        f" &nbsp;&middot;&nbsp; restored {image_mse(restored, clean):.3e}"
        f"<br>TV-L1 energy &nbsp; noisy {tvl1_energy(noisy, noisy, lam):.1f}"
        f" &nbsp;&middot;&nbsp; restored {tvl1_energy(noisy, restored, lam):.1f}"
    )


def show_lambda(lam):
    lam = float(lam)
    if lam not in tvl1_cache:
        caption.value = caption_html("solving…")
        tvl1_cache[lam] = denoise_image(noisy, lam, data="l1")
    restored = tvl1_cache[lam]
    axis_restored.cla()
    draw_gray_image(axis_restored, restored, "TV-L1")
    comparison.suptitle(rf"$\lambda = {lam:.3g}$", fontsize=14)
    comparison.canvas.draw_idle()
    caption.value = tvl1_caption(restored, lam)


def on_slider(change):
    show_lambda(change["new"])


lam_slider.observe(on_slider, names="value")
show_lambda(lam_slider.value)
controls = widgets.HBox(
    [lam_slider],
    layout=widgets.Layout(width="100%", justify_content="center", padding="0 0 6px 0"),
)
display(
    widgets.VBox(
        [controls, comparison.canvas, caption],
        layout=widgets.Layout(align_items="center", width="100%"),
    )
)

## Effect of various parameters, and TV-L1 against ROF

The plot above changes $\lambda$ on one image and one noise level. Here, we vary the image, its size, and the noise, and we compare TV-L1 with ROF on the same noisy image.

- image: the synthetic image, or one of the two natural images.
- size: the longer side of the image in pixels. The cost of a solve grows with the number of pixels: a natural image at 256 pixels takes about half a minute per model, and at 128 pixels a few seconds.
- $p$: the fraction of pixels replaced by 0 or 1.
- $\lambda$ for TV-L1 and $\lambda$ for ROF. The default for ROF, $\lambda = 3$, is the best of $\{1, 2, 3, 4, 6, 8\}$ on the synthetic image.

TV-L1 removes the impulses and keeps the edges and the contrast of each shape. ROF pays quadratically for large residuals, so it spreads each impulse into a gray blob, and a $\lambda$ small enough to hide the blobs also blurs the edges. As $p$ grows, impulses start to touch and form clusters with a smaller perimeter for their area, and TV-L1 needs a smaller $\lambda$ to remove them.

Each result is cached, so returning to a setting is instant.

In [ ]:
import ipywidgets as widgets
from IPython.display import display

with plt.ioff():
    parameter_figure = plt.figure(figsize=(12, 3.5))
parameter_axes = parameter_figure.subplots(1, 4)
parameter_figure.subplots_adjust(top=0.84, bottom=0.02, left=0.01, right=0.99, wspace=0.06)

image_cache = {}
parameter_cache = {}
caption = widgets.HTML(layout=widgets.Layout(width="100%"))
control_layout = widgets.Layout(width="250px", margin="0 8px")
control_style = {"description_width": "70px"}
image_choice = widgets.Dropdown(
    options=IMAGE_NAMES,
    value="synthetic",
    description="image",
    layout=control_layout,
    style=control_style,
)
size_choice = widgets.SelectionSlider(
    options=[64, 96, 128, 192, 256],
    value=SIZE,
    description="size",
    continuous_update=False,
    layout=control_layout,
    style=control_style,
)
fraction_slider = widgets.FloatSlider(
    value=FRACTION,
    min=0.0,
    max=0.6,
    step=0.05,
    description="p",
    continuous_update=False,
    readout_format=".2f",
    layout=control_layout,
    style=control_style,
)
lam_l1_slider = widgets.FloatLogSlider(
    value=1.0,
    base=10,
    min=-1.5,
    max=1.0,
    step=0.05,
    description="λ TV-L1",
    continuous_update=False,
    readout_format=".3g",
    layout=control_layout,
    style=control_style,
)
lam_l2_slider = widgets.FloatLogSlider(
    value=3.0,
    base=10,
    min=-1.0,
    max=1.5,
    step=0.05,
    description="λ ROF",
    continuous_update=False,
    readout_format=".3g",
    layout=control_layout,
    style=control_style,
)


def caption_html(text):
    return f"<div style='text-align:center; font-size:14px; color:#333; padding-top:4px;'>{text}</div>"


def noisy_image(name, size, fraction):
    key = (name, size, fraction)
    if key not in image_cache:
        image = load_image(name, size)
        image_cache[key] = (image, make_noisy(image, fraction))
    return image_cache[key]


def restore(name, size, fraction, data, lam):
    key = (name, size, fraction, data, lam)
    if key not in parameter_cache:
        _clean, observed = noisy_image(name, size, fraction)
        parameter_cache[key] = denoise_image(observed, lam, data=data)
    return parameter_cache[key]


def show_parameters(_change=None):
    name = image_choice.value
    size = int(size_choice.value)
    fraction = round(float(fraction_slider.value), 2)
    lam_l1 = float(lam_l1_slider.value)
    lam_l2 = float(lam_l2_slider.value)
    original, observed = noisy_image(name, size, fraction)
    caption.value = caption_html("solving TV-L1…")
    tvl1 = restore(name, size, fraction, "l1", lam_l1)
    caption.value = caption_html("solving ROF…")
    rof = restore(name, size, fraction, "l2", lam_l2)
    panels = [
        ("Clean", original),
        ("Noisy", observed),
        (rf"TV-L1, $\lambda = {lam_l1:.3g}$", tvl1),
        (rf"ROF, $\lambda = {lam_l2:.3g}$", rof),
    ]
    for axis, (title, image) in zip(parameter_axes, panels):
        axis.cla()
        draw_gray_image(axis, image, title)
    height, width = original.shape
    parameter_figure.suptitle(f"{name}, {height} × {width}, $p = {fraction:.2f}$", fontsize=14)
    parameter_figure.canvas.draw_idle()
    caption.value = caption_html(
        f"MSE &nbsp; noisy {image_mse(observed, original):.3e}"
        f" &nbsp;&middot;&nbsp; TV-L1 {image_mse(tvl1, original):.3e}"
        f" &nbsp;&middot;&nbsp; ROF {image_mse(rof, original):.3e}"
    )


for control in (image_choice, size_choice, fraction_slider, lam_l1_slider, lam_l2_slider):
    control.observe(show_parameters, names="value")
show_parameters()
row_layout = widgets.Layout(width="100%", justify_content="center", align_items="center")
controls = widgets.VBox(
    [
        widgets.HBox([image_choice, size_choice, fraction_slider], layout=row_layout),
        widgets.HBox([lam_l1_slider, lam_l2_slider], layout=row_layout),
    ],
    layout=widgets.Layout(width="100%", padding="0 0 6px 0"),
)
display(
    widgets.VBox(
        [controls, parameter_figure.canvas, caption],
        layout=widgets.Layout(align_items="center", width="100%"),
    )
)

## Figure 4 of Chambolle and Pock

Figure 4 of [1] denoises the full 375 × 500 Louvre image with 25% salt and pepper, using $\lambda = 1.5$ for TV-L1 and $\lambda = 8$ for ROF. A solve at this size takes over a minute, so we show the saved result of the experiment script rather than recompute it here:

```bash
PYTHONPATH=src python -m experiments.exp_tvl1.run_exp_tvl1 \
    --image models/experiments/exp_tvl1/louvre.png --size 500 \
    --lam-l1 1.5 --lam-l2 8 \
    --output results/experiments/exp_tvl1/louvre/exp_tvl1_comparison.png --no-show
```

| | MSE |
|---|---|
| noisy | 8.01e-02 |
| TV-L1, $\lambda = 1.5$ | 4.2e-03 |
| ROF, $\lambda = 8$ | 2.4e-02 |

![Louvre](../results/experiments/exp_tvl1/louvre/exp_tvl1_comparison.png)

TV-L1 removes almost all of the noise and keeps the glass grid of the pyramid and the small blocks in the foreground, as in Figure 4(d) of [1]. At $\lambda = 8$, our ROF result still shows many impulses. Figure 4(c) of [1] is much smoother at the same $\lambda$, so the paper's ROF data weight is probably scaled differently from our `data="l2"` term.

## Conclusion

We replaced the quadratic data term of ROF with an $\ell_1$ term, which changes only the primal step of the Chambolle-Pock iteration into a soft threshold. Written as a graph, the pixel grid lets the point cloud solver run the TV-L1 model of [1] unchanged. TV-L1 removes salt-and-pepper noise outright and keeps the contrast of what remains, while ROF can only spread the impulses into blurs.

## References

[1] A. Chambolle and T. Pock, A first-order primal-dual algorithm for convex problems with applications to imaging, Journal of Mathematical Imaging and Vision 40 (2011), 120–145.

[2] D. Tenbrinck, F. Lozes, and A. Elmoataz, Solving minimal surface problems on surfaces and point clouds, Scale Space and Variational Methods in Computer Vision (2015).

[3] T. F. Chan and S. Esedoglu, Aspects of total variation regularized $L^1$ function approximation, SIAM Journal on Applied Mathematics 65 (2005), 1817–1837.